In [ ]:
import os, sys
import glob
import math
import cv2
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.spatial import KDTree

In [ ]:
# input_path: 1000×1000 pixel image files which were generated by Analysis/Pathological_regions/01.inputImage.ipynb
def get_boundary_image(input_path, output_dir):
    png_name = os.path.basename(input_path).replace('.png','')
    image = cv2.imread(input_path)
    if image is None:
        sys.exit('Could not read the image.')

    height, width = image.shape[:2]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)
    kernel = np.ones((3, 3), np.uint8)
    edges = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_image = np.zeros_like(image)
    cv2.drawContours(boundary_image, contours, -1, (255, 255, 255), 1)
    cv2.imwrite(os.path.join(output_dir, f"{png_name}_boundary_image.png"), boundary_image)

In [ ]:
# Part 1: Identify boundary bins and extract their bin100 coordinates.

In [ ]:
sample = 'sample_tmp'
boundary_type = 'RA'
h5ad_path = f'/data/work/STAGATE/result/{sample}/tissue_bin100/{sample}_bin100.h5ad'
# Stereo-seq h5ad data containing STAGATE annotations
imagePath = f'/data/work/pathologicalRegion_border/border_pic/{boundary_type}/{sample}.Annihilated.border._boundary_image.png'
# Boundary images, you can generate them using the get_boundary_image function above

In [29]:
adata = sc.read_h5ad(h5ad_path)
adata_xypos = pd.DataFrame(adata.obsm['spatial'],index = adata.obs_names)
adata_xypos = adata_xypos.rename(columns={0: 'y', 1: 'x'})

In [ ]:
msk = np.array(Image.open(imagePath)) / 255.0 
s = msk.shape
if s[0] != s[1] or s[0] != 1000:
    print(f"{imagePath} Wrong picture sizes", file=sys.stderr)

r1 = (adata_xypos['x'].min(), adata_xypos['x'].max())
r2 = (adata_xypos['y'].min(), adata_xypos['y'].max())
if r1[1] - r1[0] > r2[1] - r2[0]:
    k = (r1[1] - r1[0]) / s[0]
else:
    k = (r2[1] - r2[0]) / s[1]

In [ ]:
white_coords = np.argwhere(msk.mean(axis=2) > 0.9)
bin_coords_pixel = np.column_stack([
    s[0] - 1 - ((adata_xypos['y'] - r2[0]) / k).round(), 
    ((adata_xypos['x'] - r1[0]) / k).round()  
]) 
tree = KDTree(white_coords) 
distances, _ = tree.query(bin_coords_pixel, k=1) 

In [ ]:
bin_size_pixels = 2 
adata_xypos['is_boundary'] = distances <= bin_size_pixels
adata_xypos.to_csv(f"/data/work/pathologicalRegion_border/border_pic/result/{boundary_type}/{sample}.border.csv")
boundary_csv = adata_xypos[adata_xypos['is_boundary']]
boundary_csv.to_csv(f"/data/work/pathologicalRegion_border/border_pic/result/{boundary_type}/{sample}.boundary.csv")

plt.figure(figsize=(10, 10))
plt.scatter(adata_xypos['x'], adata_xypos['y'], c=np.where(adata_xypos['is_boundary'], '#FFDD00', '#EEEEEE'), 
            s=15, marker='.')
plt.axis('equal')
plt.axis('off')
plt.savefig(f"/data/work/pathologicalRegion_border/border_pic/result/{boundary_type}/{sample}.border.png", dpi=100, bbox_inches='tight')
plt.close()

In [ ]:
# Part 2: Prepare boundary and subMP coordinates for distance analysis.

In [ ]:
output_path = '/data/work/pathologicalRegion_border/border_pic/bar/data'
boundary_type = 'RA'

sample = 'sample_tmp'
h5ad_path = f'/data/work/STAGATE/result/{sample}/tissue_bin100/{sample}_bin100.h5ad'
boundaryPath = glob.glob(f'/data/work/pathologicalRegion_border/border_pic/result/{boundary_type}/{sample}.boundary*.csv')

In [64]:
adata = sc.read_h5ad(h5ad_path)

In [65]:
dfs = []
for filePath in boundaryPath:
    boundaryFile = pd.read_csv(filePath,index_col=0)
    dfs.append(boundaryFile)
merged_df = pd.concat(dfs)
merged_df.to_csv(f'{output_path}/{sample}.{boundary_type}.xy.csv')

In [ ]:
adata_xypos = pd.DataFrame(adata.obsm['spatial'],index = adata.obs_names)
adata_xypos = adata_xypos.rename(columns={0: 'y', 1: 'x'})
adata_xypos['bin100_pos']= adata_xypos.index

adata.obs['subMP'] = adata.obs['subMP'].astype(str)
adata.obs['STAGATE'] = adata.obs['STAGATE'].astype(str)
unique_values = adata.obs['STAGATE'].unique()
subMP = adata.obs.loc[~adata.obs['subMP'].isin(['Undefined', 'unknown']),['subMP']].copy()
subMP['bin100_pos']= subMP.index
subMP_xy=pd.merge(subMP,adata_xypos,how='left',on=['bin100_pos']).set_index('bin100_pos')
subMP_xy.to_csv(f'{output_path}/{sample}.subMP.xy.csv')

In [ ]:
# Part 3: Calculate distances from subMP bins to the nearest boundary bins.

In [1]:
import numpy as np
from scipy.spatial import cKDTree 

In [ ]:
def calculate_distances(boundary_points, subMP_points):
    tree = cKDTree(boundary_points)
    distances, _ = tree.query(subMP_points)
    return np.mean(distances), distances

In [ ]:
boundary_paths=glob.glob(f'/data/work/pathologicalRegion_border/border_pic/bar/data/*.xy.csv')
sample_list=[]
for boundary_path in boundary_paths:
    sample = os.path.basename(boundary_path).split('.')[0]
    sample_list.append(sample)
unique_samples = list(set(sample_list))

In [ ]:
results = []
output_path=f'/data/work/pathologicalRegion_border/border_pic/bar/data/All.distance_mean.new.csv'
for sample in unique_samples:
    boundary_paths=glob.glob(f'/data/work/pathologicalRegion_border/border_pic/bar/data/{sample}.R*.xy.csv')
    subMP_path=f'/data/work/pathologicalRegion_border/border_pic/bar/data/{sample}.subMP.xy.csv'
    for boundary_path in boundary_paths:
        boundary_type = os.path.basename(boundary_path).split('.')[1]
        boundary_file = pd.read_csv(boundary_path,index_col=0)
        subMP_file = pd.read_csv(subMP_path,index_col=0)
    
        subMP_list=subMP_file['subMP'].unique().tolist()
        for subMP in subMP_list:
            single_subMP = subMP_file.loc[subMP_file['subMP']==subMP,]
            boundary_points = boundary_file[['x', 'y']].values
            subMP_points = single_subMP[['x', 'y']].values
            mean_dist, all_dists = calculate_distances(boundary_points, subMP_points)
            median_dist = np.median(all_dists)
            std_dist = np.std(all_dists)
            results.append({
                'sample': sample,
                'boundary_type': boundary_type,
                'subMP': subMP,
                'mean_distance': mean_dist,
                'median_distance': median_dist,
                'distance_std': std_dist,
                'min_distance': np.min(all_dists),
                'max_distance': np.max(all_dists),
                'boundary_points': len(boundary_file),
                'subMP_points': len(single_subMP)
            })
result_df = pd.DataFrame(results)
result_df.to_csv(output_path, index=False)   

In [145]:
all_data = pd.read_csv('/data/work/pathologicalRegion_border/border_pic/bar/data/All.distance_mean.new.csv')
RR_data = all_data.loc[all_data['boundary_type']=='RR',]
RR_data.to_csv('/data/work/pathologicalRegion_border/border_pic/bar/data/RR.distance_mean.new.csv', index=False)  
RA_data = all_data.loc[all_data['boundary_type']=='RA',]
RA_data.to_csv('/data/work/pathologicalRegion_border/border_pic/bar/data/RA.distance_mean.new.csv', index=False)
RF_data = all_data.loc[all_data['boundary_type']=='RF',]
RF_data.to_csv('/data/work/pathologicalRegion_border/border_pic/bar/data/RF.distance_mean.new.csv', index=False)